# Gamma Law Simulation

This notebook studies the Gamma distribution from three complementary angles:

1. direct simulation with NumPy,
2. interactive exploration of the parameters with Plotly,
3. simulation by rejection, followed by a discussion of the method's limitations.

In [8]:
import numpy as np
import pandas as pd
from scipy import special, stats
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(12345)

## 1. Direct simulation with NumPy

We use the shape-scale parameterization of the Gamma law. For a shape parameter $k > 0$ and a scale parameter $\theta > 0$, the density is

$$
f(x; k, \theta) = \frac{x^{k-1} e^{-x / \theta}}{\Gamma(k) \theta^k}, \qquad x > 0.
$$

Here $\Gamma(k)$ is the Gamma function, defined for $k > 0$ by

$$
\Gamma(k) = \int_0^{\infty} t^{k-1} e^{-t} \, dt.
$$

It generalizes the factorial, since $\Gamma(n) = (n-1)!$ for every integer $n \ge 1$. In the density above, it plays the role of a normalizing constant ensuring that the total area under the curve is equal to 1.

Its first two moments are

$$
\mathbb{E}[X] = k\theta, \qquad \mathrm{Var}(X) = k\theta^2.
$$

NumPy uses exactly this parameterization through `rng.gamma(shape=k, scale=theta, size=...)`.
If one prefers the shape-rate notation with rate $\beta > 0$, then $\theta = 1 / \beta$.

In [10]:
shape = 2.5
scale = 1.2
size = 20_000

gamma_numpy = rng.gamma(shape=shape, scale=scale, size=size)

summary_numpy = pd.DataFrame(
    {
        "quantity": ["mean", "variance"],
        "theoretical": [shape * scale, shape * scale**2],
        "empirical": [gamma_numpy.mean(), gamma_numpy.var(ddof=1)],
    }
)

summary_numpy

,quantity,theoretical,empirical
0,mean,3.0,3.017334
1,variance,3.6,3.578257


## 2. Interactive parameter exploration with Plotly

The interactive figure below compares a simulated histogram with the theoretical Gamma density.
Two sliders control the shape parameter $k$ and the scale parameter $\theta$ directly on the interval $[0.1, 3.0]$.

In the code:

- `stats.gamma.pdf(x, a=k, scale=theta)` evaluates the probability density function pointwise.
- `stats.gamma.ppf(q, a=k, scale=theta)` returns the quantile associated with probability level `q`, that is, the inverse cumulative distribution function.

Here we use `ppf(0.999, ...)` to choose an upper bound for the horizontal axis that contains almost all of the distribution mass.

In [ ]:
shape_slider = widgets.FloatSlider(
    value=1.5,
    min=0.1,
    max=3.0,
    step=0.1,
    description="shape k",
    continuous_update=False,
    readout_format=".1f",
)

scale_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=3.0,
    step=0.1,
    description="scale",
    continuous_update=False,
    readout_format=".1f",
)

def update_gamma_plot(shape, scale):

    sample = rng.gamma(shape=shape, scale=scale, size=5_000)
    x_max = stats.gamma.ppf(0.999, a=shape, scale=scale)

    if not np.isfinite(x_max) or x_max <= 0.0:
        x_max = max(sample.max(), shape * scale)

    x = np.linspace(0.0, x_max, 500)
    y = stats.gamma.pdf(x, a=shape, scale=scale)

    fig = go.Figure()
    fig.add_histogram(
        x=sample,
        histnorm="probability density",
        nbinsx=60,
        name="NumPy sample",
        opacity=0.65,
        marker_color="#4C78A8",
    )
    fig.add_scatter(
        x=x,
        y=y,
        mode="lines",
        name="Theoretical density",
        line=dict(color="#E45756", width=3),
    )
    fig.update_layout(
        title=f"Gamma distribution: shape k={shape:.1f}, scale theta={scale:.1f}",
        template="plotly_white",
        barmode="overlay",
        xaxis_title="x",
        yaxis_title="density",
        xaxis=dict(range=[0.0, 1.05 * x_max]),
        width=900,
        height=500,
    )
    fig.show()


interactive_output = widgets.interactive_output(
    update_gamma_plot,
    {"shape": shape_slider, "scale": scale_slider},
)

display(widgets.VBox([widgets.HBox([shape_slider, scale_slider]), interactive_output]))

## 3. Rejection-based simulation

For $k \geq 1$, we use an exponential proposal distribution

$$
g(x) = \frac{1}{k\theta} e^{-x / (k\theta)}, \qquad x > 0.
$$

A direct calculation shows that

$$
f(x; k, \theta) \leq M_k g(x), \qquad M_k = \frac{k^k e^{1-k}}{\Gamma(k)}.
$$

If $Y \sim g$ and $U \sim \mathcal{U}(0,1)$ are independent, then the rejection test can be written in the numerically stable log form

$$
\log U \leq (k-1)\left[\log\!\left(\frac{Y}{k\theta}\right) + 1 - \frac{Y}{k\theta}\right].
$$

For $0 < k < 1$, pure rejection with this proposal is not convenient. A standard workaround is the boosting identity:

$$
X = Y U^{1/k}, \qquad Y \sim \Gamma(k+1, \theta), \quad U \sim \mathcal{U}(0,1),
$$

which reduces the problem to simulating a Gamma law with shape parameter $k+1 > 1$.

The following plot visualizes the rejection envelope for the case $k \geq 1$. It compares the target Gamma density $f(x;k,\theta)$ with the majorizing curve $M_k g(x)$, and shows the acceptance ratio $f(x;k,\theta)/(M_k g(x))$ on the same grid of $x$ values.

In [37]:
from plotly.subplots import make_subplots

shape_envelope = 2.5
scale_envelope = 1.2

x_envelope = np.linspace(
    0.0,
    stats.gamma.ppf(0.999, a=shape_envelope, scale=scale_envelope),
    600,
)

target_density = stats.gamma.pdf(
    x_envelope,
    a=shape_envelope,
    scale=scale_envelope,
)

proposal_density = stats.expon.pdf(
    x_envelope,
    scale=shape_envelope * scale_envelope,
)

majorization_constant = (
    shape_envelope**shape_envelope
    * np.exp(1.0 - shape_envelope)
    / special.gamma(shape_envelope)
)

majorizing_density = majorization_constant * proposal_density
acceptance_ratio = target_density / majorizing_density

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.5, 0.5],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Target density and rejection envelope",
        "Acceptance ratio",
    ),
)

fig.add_scatter(
    x=x_envelope,
    y=target_density,
    mode="lines",
    name="target f(x; k, theta)",
    line=dict(color="#E45756", width=3),
    row=1,
    col=1,
)

fig.add_scatter(
    x=x_envelope,
    y=majorizing_density,
    mode="lines",
    name="majorant M_k g(x)",
    line=dict(color="#4C78A8", width=3, dash="dash"),
    row=1,
    col=1,
)

fig.add_scatter(
    x=x_envelope,
    y=acceptance_ratio,
    mode="lines",
    name="f(x; k, theta) / (M_k g(x))",
    line=dict(color="#54A24B", width=3),
    row=1,
    col=2,
)

fig.add_hline(
    y=1.0,
    line_dash="dot",
    line_color="#888888",
    row=1,
    col=2,
)

fig.update_layout(
    title=(
        f"Gamma rejection envelope: k={shape_envelope}, "
        f"theta={scale_envelope}, M_k={majorization_constant:.3f}"
    ),
    template="plotly_white",
    width=800,
    height=430,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.28,
        xanchor="center",
        x=0.5,
    ),
)

fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)

fig.update_yaxes(title_text="density", row=1, col=1)
fig.update_yaxes(title_text="ratio", range=[0, 1.05], row=1, col=2)

fig.show()

In [26]:
def gamma_rejection_ge_one(shape, scale, size, rng):
    if shape < 1.0:
        raise ValueError("shape must be at least 1 in gamma_rejection_ge_one")

    accepted_chunks = []
    accepted_total = 0
    proposals_total = 0

    log_M = shape * np.log(shape) + 1.0 - shape - special.gammaln(shape)
    approx_acceptance = float(np.exp(-log_M))
    approx_acceptance = max(min(approx_acceptance, 1.0), 1e-3)

    while accepted_total < size:
        remaining = size - accepted_total
        batch_size = max(256, int(np.ceil(1.25 * remaining / approx_acceptance)))

        proposal = rng.exponential(scale=shape * scale, size=batch_size)
        normalized = proposal / (shape * scale)
        log_acceptance = (shape - 1.0) * (np.log(normalized) + 1.0 - normalized)
        uniforms = rng.random(batch_size)
        accepted = proposal[np.log(uniforms) <= log_acceptance]

        if accepted.size > 0:
            accepted_chunks.append(accepted)
            accepted_total += accepted.size

        proposals_total += batch_size

    sample = np.concatenate(accepted_chunks)[:size]
    return sample, proposals_total


def gamma_rejection(shape, scale, size, rng):
    if shape <= 0.0 or scale <= 0.0:
        raise ValueError("shape and scale must be strictly positive")

    if shape >= 1.0:
        sample, proposals = gamma_rejection_ge_one(shape, scale, size, rng)
        return sample, {"proposals": proposals, "acceptance_rate": size / proposals}

    boosted_sample, proposals = gamma_rejection_ge_one(shape + 1.0, scale, size, rng)
    uniforms = rng.random(size)
    sample = boosted_sample * uniforms ** (1.0 / shape)
    return sample, {"proposals": proposals, "acceptance_rate": size / proposals}

In [27]:
shape_reject = 0.7
scale_reject = 1.5
size_reject = 20_000

gamma_reject_sample, reject_info = gamma_rejection(
    shape=shape_reject,
    scale=scale_reject,
    size=size_reject,
    rng=rng,
)

summary_reject = pd.DataFrame(
    {
        "quantity": ["mean", "variance", "acceptance_rate"],
        "theoretical": [
            shape_reject * scale_reject,
            shape_reject * scale_reject**2,
            np.nan,
        ],
        "empirical": [
            gamma_reject_sample.mean(),
            gamma_reject_sample.var(ddof=1),
            reject_info["acceptance_rate"],
        ],
    }
)

summary_reject

,quantity,theoretical,empirical
0,mean,1.050,1.048380
1,variance,1.575,1.565188
2,acceptance_rate,NaN,0.593912


In [28]:
x = np.linspace(0.0, stats.gamma.ppf(0.999, a=shape_reject, scale=scale_reject), 500)

fig = go.Figure()
fig.add_histogram(
    x=gamma_reject_sample,
    histnorm="probability density",
    nbinsx=70,
    name="Rejection sample",
    opacity=0.65,
    marker_color="#72B7B2",
)
fig.add_scatter(
    x=x,
    y=stats.gamma.pdf(x, a=shape_reject, scale=scale_reject),
    mode="lines",
    name="Theoretical density",
    line=dict(color="#E45756", width=3),
)
fig.update_layout(
    title=(
        f"Rejection-based Gamma simulation: k={shape_reject}, "
        f"theta={scale_reject}, acceptance={reject_info['acceptance_rate']:.3f}"
    ),
    xaxis_title="x",
    yaxis_title="density",
    barmode="overlay",
    template="plotly_white",
)
fig.show()

In [ ]:
shapes_to_test = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
acceptance_rows = []

for shape_value in shapes_to_test:
    _, info = gamma_rejection(shape=shape_value, scale=1.0, size=5_000, rng=rng)
    acceptance_rows.append(
        {
            "shape": shape_value,
            "acceptance_rate": info["acceptance_rate"],
        }
    )

pd.DataFrame(acceptance_rows)

## 4. Limitations of the rejection method

This implementation is intentionally simple and pedagogical, but it has several limitations:

- It is not the most efficient Gamma sampler available. In practice, NumPy's built-in generator is faster and more robust.
- For large shape parameters, the acceptance rate decreases, so the rejection loop spends more time generating proposals that are eventually discarded.
- For very small shape parameters, the boosting step $U^{1/k}$ may become numerically delicate because much of the mass is concentrated near zero.
- The current implementation is written in Python with batched loops. It is suitable for illustration, but not ideal for large-scale Monte Carlo workloads.
- More refined algorithms, such as Marsaglia-Tsang type samplers, usually provide better performance and are preferred in production code.